# 07b — M3 (wavelet): PTBXL single-dataset detector

Trains on **ptbxl** (folds 1-8), validates same-dataset (**fold 9**), and measures **cross-dataset transfer** to **ningbo** (folds 1-8) to quantify the batch effect. **Reuses the shared `m3_features.csv` (NO re-extraction).** Localization + delineation-free QRS-localized pool (energy dropped). XGBoost only, NaN-native, Platt on native OOF.

**Judge = pooled OOF AP (folds 1-8, stable; fold 9 = ptbxl validation, only 6-7 WPW = high variance). Fold 10 (both datasets) + fold 9 of ningbo NEVER touched.**

### Selection note (differs from M1/M2 — on purpose, same rule as M3 combined 07a)

Single-dataset M3 has very few training WPW (57-58), so **`AP_train` saturates** and the train-gap is not a valid overfit probe. Selection = **max OOF AP, depth open (2/3/4)**, then a **parsimony tiebreak among true OOF-ties** (smallest K), with **inter-fold OOF AP variance + fold 9** as honest guards. M1/M2 standalone use a different rule (max-OOF-under-gap / 1-SE); M3 is consistent with its own combined detector (07a). Never selects on fold 9; final external estimate = fold 10 (ensemble notebook).

### Block 7b.0: Setup + load features (single dataset, reuse m3_features.csv)

In [ ]:
# M3 (wavelet) - SINGLE-DATASET detector. Trains on ONE dataset (folds 1-8), evaluates same-dataset
# (fold 9) AND cross-dataset transfer to the OTHER dataset (folds 1-8) to quantify the batch effect.
# Features reloaded from the shared m3_features.csv (NO re-extraction). Localization+QRS pool (energy
# dropped). Judge: AP. Fold 10 (both datasets) + fold 9 of the cross dataset NEVER touched.
import os, sys, json
import numpy as np, pandas as pd
ROOT      = r'.'
PROCESSED = os.path.join(ROOT,'data','processed')
SRC       = os.path.join(ROOT,'src')
FIG       = os.path.join(ROOT,'reports','figures')
METRICS   = os.path.join(ROOT,'reports','metrics')
sys.path.insert(0, SRC)
from evaluation import evaluate_standard
TRAIN_SRC, CROSS_SRC = 'ptbxl', 'ningbo'    # 07b = PTB detector  (07c uses: 'ningbo','ptbxl')
TAG = TRAIN_SRC
POOL = 'localization'                       # drop energy/entropy (ablation: dead weight)
TIE_EPS = 0.01                              # parsimony tiebreak band on OOF AP (true ties -> smallest K)
FEATURES_CSV = os.path.join(PROCESSED,'m3_features.csv')
METAC = ['ecg_id','patient_id','label','fold','source','extraction_failed']
dfall = pd.read_csv(FEATURES_CSV, dtype={'ecg_id':str})
df  = dfall[dfall.source==TRAIN_SRC].reset_index(drop=True)     # training data (same)
dfx = dfall[dfall.source==CROSS_SRC].reset_index(drop=True)     # cross-dataset data
ALL_FEATS = [c for c in dfall.columns if c not in METAC]
print(f"M3 single-dataset detector: train={TRAIN_SRC} | cross-test={CROSS_SRC} | POOL={POOL}")
print(f"{TRAIN_SRC}: {len(df)} ECGs, {int((df.label==1).sum())} WPW | WPW/fold {df[df.label==1].groupby('fold').size().to_dict()}")
print(f"{CROSS_SRC} (cross): {len(dfx)} ECGs, {int((dfx.label==1).sum())} WPW")
print(f"Available features: {len(ALL_FEATS)} (qrs-localized {sum('qrs' in c for c in ALL_FEATS)})")
print(f"Train(1-8) WPW: {int(df[(df.fold.between(1,8))&(df.label==1)].shape[0])} | "
      f"Val(9): {int(df[(df.fold==9)&(df.label==1)].shape[0])} | Test(10): {int(df[(df.fold==10)&(df.label==1)].shape[0])} [UNTOUCHED]")

### Block 7b.0b: Patient-leakage check (blocking assert)

In [ ]:
# Patient-leakage check within the training dataset (blocking assert).
pf = df.groupby('patient_id')['fold'].nunique(); leaky = pf[pf>1]
print(f"{TRAIN_SRC}: {df.patient_id.nunique()} patients, {len(df)} ECGs | in >1 fold: {len(leaky)}")
assert len(leaky)==0, f"PATIENT LEAK {TRAIN_SRC}: {len(leaky)} patients in multiple folds."
print("[OK] No patient leakage.")

### Block 7b.2: Single-dataset FDR gate + localization pool + dedup>0.95

In [ ]:
# Feature selection on TRAIN_SRC folds 1-8: SINGLE-DATASET gate |Cohen's d|>0.3 AND p_FDR<0.05 (BH) AND
# IC95 bootstrap of d excludes 0 (NO cross-dataset criterion - single source). FINAL pool = localization
# (energy/entropy dropped). Fast Spearman>0.95 dedup.
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm
SEL_CSV = os.path.join(PROCESSED, f'm3_{TAG}_selection.csv')
tr = df[df.fold.between(1,8)]
print(f"Train folds 1-8 ({TRAIN_SRC}): {len(tr)} ECGs, {int((tr.label==1).sum())} WPW")
def cohens_d(a,b):
    a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return np.nan
    sp=np.sqrt(((len(a)-1)*a.var(ddof=1)+(len(b)-1)*b.var(ddof=1))/(len(a)+len(b)-2))
    return (a.mean()-b.mean())/sp if sp>0 else np.nan
def d_ci(a,b,n=1000,seed=42):
    rng=np.random.default_rng(seed); a,b=a[~np.isnan(a)],b[~np.isnan(b)]
    if len(a)<2 or len(b)<2: return (np.nan,np.nan)
    ds=[cohens_d(rng.choice(a,len(a),True),rng.choice(b,len(b),True)) for _ in range(n)]
    return tuple(np.nanpercentile(ds,[2.5,97.5]))
if os.path.exists(SEL_CSV) and set(pd.read_csv(SEL_CSV,usecols=['feature'])['feature'])==set(ALL_FEATS):
    res=pd.read_csv(SEL_CSV); print(f"[guard] {os.path.basename(SEL_CSV)} reloaded ({len(res)} features).")
else:
    w,nn=tr[tr.label==1],tr[tr.label==0]; rows=[]
    for c in tqdm(ALL_FEATS, desc='Gate (d+p+IC)', unit='feat'):
        a,b=w[c].values.astype(float),nn[c].values.astype(float); d=cohens_d(a,b)
        try: _,p=mannwhitneyu(a[~np.isnan(a)],b[~np.isnan(b)],alternative='two-sided')
        except Exception: p=np.nan
        lo,hi=d_ci(a,b); ci_ok=(np.isfinite(lo) and np.isfinite(hi) and (lo>0)==(hi>0))
        rows.append({'feature':c,'d':d,'p_raw':p,'ci_excl0':ci_ok})
    res=pd.DataFrame(rows); ok=res.p_raw.notna()
    res.loc[ok,'p_FDR']=multipletests(res.loc[ok,'p_raw'],method='fdr_bh')[1]
    res['gate']=(res.d.abs()>0.3)&(res.p_FDR<0.05)&res.ci_excl0
    res=res.sort_values('d',key=lambda s:s.abs(),ascending=False).reset_index(drop=True)
    res.to_csv(SEL_CSV,index=False)
passed=res[res.gate].feature.tolist()
def _isenergy(c): return ('energy' in c) or ('wentropy' in c)
pool_feats=[c for c in passed if not _isenergy(c)] if POOL=='localization' else list(passed)
print(f"Pass the gate: {len(passed)} | POOL='{POOL}' -> pool: {len(pool_feats)} (qrs-localized {sum('qrs' in c for c in pool_feats)})")
def dedup_fast(feats,thr=0.95):
    sub=tr[feats].rank().to_numpy(); cm=np.nanmean(sub,axis=0)
    ii=np.where(np.isnan(sub)); sub[ii]=np.take(cm,ii[1])
    C=np.abs(np.corrcoef(sub,rowvar=False)); idx={f:i for i,f in enumerate(feats)}; keep=[]
    for f in feats:
        if all(C[idx[f],idx[k]]<=thr for k in keep): keep.append(f)
    return keep
dedup_list=dedup_fast(pool_feats,0.95) if pool_feats else []
print(f"After dedup>0.95: {len(dedup_list)} features (= k_max) (qrs-localized {sum('qrs' in c for c in dedup_list)})\n")
pd.set_option('display.float_format',lambda x:f'{x:.3f}')
print(res[res.feature.isin(dedup_list)][['feature','d','p_FDR']].head(15).to_string(index=False))

### Block 7b.3: AP-vs-k same (+95% CI) + cross-dataset batch-effect curve

In [ ]:
# AP-vs-k: AP OOF same-dataset (provisional K, 95% plateau) + bootstrap 95% CI band + AP/AUC cross-dataset
# (batch-effect viz: AP collapses, AUC holds). step 2; cached. The provisional K seeds the 7x.3d grid.
from sklearn.metrics import average_precision_score, roc_auc_score
from xgboost import XGBClassifier
from tqdm import tqdm
import matplotlib.pyplot as plt
RK_CSV = os.path.join(PROCESSED, f'm3_{TAG}_ap_vs_k.csv')
CI_CSV = os.path.join(PROCESSED, f'm3_{TAG}_ap_vs_k_ci.csv')
d8 = df[df.fold.between(1,8)].reset_index(drop=True); y, folds = d8.label.values, d8.fold.values
FOLD_MASKS=[(folds!=h,folds==h) for h in sorted(np.unique(folds))]
spw8=(y==0).sum()/max((y==1).sum(),1)
dcx = dfx[dfx.fold.between(1,8)].reset_index(drop=True); ycx=dcx.label.values   # cross folds 1-8 (cross 9-10 untouched)
def make_xgb(spw=None,**kw):
    if spw is None: spw=spw8
    p=dict(n_estimators=100,max_depth=2,learning_rate=0.05,subsample=0.8,colsample_bytree=0.8,
           reg_lambda=2.0,min_child_weight=3,scale_pos_weight=spw,eval_metric='aucpr',
           tree_method='hist',random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
def oof_same(feats,**kw):
    X=d8[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y[trm].sum()==0 or y[vam].sum()==0: continue
        spw=(y[trm]==0).sum()/max((y[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**kw).fit(X[trm],y[trm]).predict_proba(X[vam])[:,1]
    return oof
def ap_boot(yy,pp,n=200,seed=42):
    rng=np.random.default_rng(seed); pos,neg=np.where(yy==1)[0],np.where(yy==0)[0]; a=np.empty(n)
    for i in range(n):
        idx=np.concatenate([rng.choice(pos,len(pos),True),rng.choice(neg,len(neg),True)]); a[i]=average_precision_score(yy[idx],pp[idx])
    return np.median(a),np.percentile(a,2.5),np.percentile(a,97.5)
k_max=len(dedup_list)
if os.path.exists(RK_CSV):
    rk=pd.read_csv(RK_CSV); print(f"[guard] {os.path.basename(RK_CSV)} reloaded ({len(rk)} k).")
else:
    rows=[]
    for k in tqdm(range(1,k_max+1,2), desc='k (same+cross)', unit='k'):
        feats=dedup_list[:k]; oof=oof_same(feats); msk=~np.isnan(oof)
        ap_same=average_precision_score(y[msk],oof[msk]); auc_same=roc_auc_score(y[msk],oof[msk])
        mdl=make_xgb().fit(d8[feats],y); pc=mdl.predict_proba(dcx[feats])[:,1]
        rows.append({'k':k,'AP_same':ap_same,'AUC_same':auc_same,
                     'AP_cross':average_precision_score(ycx,pc),'AUC_cross':roc_auc_score(ycx,pc)})
        if k%21==0: pd.DataFrame(rows).to_csv(RK_CSV,index=False)
    rk=pd.DataFrame(rows); rk.to_csv(RK_CSV,index=False)
if os.path.exists(CI_CSV):
    rkci=pd.read_csv(CI_CSV); print(f"[guard] {os.path.basename(CI_CSV)} reloaded.")
else:
    ks_ci=sorted(set(list(range(20,k_max+1,40))+[k_max])); rows=[]
    for k in tqdm(ks_ci, desc='AP-same bootstrap CI', unit='k'):
        oof=oof_same(dedup_list[:k]); m=~np.isnan(oof); med,lo,hi=ap_boot(y[m],oof[m],n=200)
        rows.append({'k':k,'AP':med,'lo':lo,'hi':hi})
    rkci=pd.DataFrame(rows); rkci.to_csv(CI_CSV,index=False)
fig,ax=plt.subplots(figsize=(13,6))
ax.fill_between(rkci.k,rkci.lo,rkci.hi,color='#2563eb',alpha=.15,label='95% bootstrap CI (AP same)')
ax.plot(rk.k,rk.AP_same,'-',color='#2563eb',lw=1,label=f'AP OOF same ({TRAIN_SRC}) - chooses k')
ax.plot(rk.k,rk.AP_cross,'-',color='#dc2626',lw=1,label=f'AP cross ({CROSS_SRC}) - collapses')
ax.plot(rk.k,rk.AUC_cross,'--',color='#16a34a',lw=1,label=f'AUC cross ({CROSS_SRC}) - holds (detection transfers)')
ax.set(xlabel='k',ylabel='metric'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'M3 {TRAIN_SRC} - AP same (+95% CI) vs AP/AUC cross ({CROSS_SRC})')
plt.savefig(os.path.join(FIG,f'm3_{TAG}_ap_vs_k.png'),dpi=150,bbox_inches='tight'); plt.show()
kbest=rk.loc[rk.AP_same.idxmax()]; K_auto=int(rk[rk.AP_same>=0.95*kbest.AP_same].k.min())
print(f"\nMax AP same at k={int(kbest.k)} (AP={kbest.AP_same:.3f}); 95% plateau from k={K_auto} (provisional).")
print(f"At k={K_auto}: cross AP {rk.iloc[(rk.k-K_auto).abs().idxmin()].AP_cross:.3f} vs same -> collapse = batch effect.")

### Block 7b.3c: Overfit diagnostic — AP_train saturation (depth 2/3/4); defines diag_one

In [ ]:
# Overfit diagnostic. Defines diag_one. AP_train saturates (~1.0) with scarce positives (57-58 WPW) ->
# train-gap invalid; judge on OOF + fold9 + inter-fold variance. Depth 2/3/4 at K_auto. Cached.
GRIDC_CSV = os.path.join(PROCESSED, f'm3_{TAG}_hpgrid_diag.csv')
y8 = d8.label.values
f9 = df[df.fold==9].reset_index(drop=True); y9 = f9.label.values
def diag_one(cfg, feats):
    X8=d8[feats]; X9=f9[feats]; oof=np.full(len(d8),np.nan)
    for trm,vam in FOLD_MASKS:
        if y8[trm].sum()==0 or y8[vam].sum()==0: continue
        spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
        oof[vam]=make_xgb(spw,**cfg).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
    m=~np.isnan(oof); ao=average_precision_score(y8[m],oof[m])
    mdl=make_xgb(**cfg).fit(X8,y8); at=average_precision_score(y8,mdl.predict_proba(X8)[:,1])
    af=average_precision_score(y9,mdl.predict_proba(X9)[:,1]) if y9.sum()>0 else np.nan
    return ao,at,at-ao,af
K_DIAG=K_auto; FEATS_DIAG=dedup_list[:K_DIAG]
if os.path.exists(GRIDC_CSV):
    G=pd.read_csv(GRIDC_CSV); print(f"[guard] {os.path.basename(GRIDC_CSV)} reloaded.")
else:
    GRID=[dict(max_depth=d,learning_rate=lr,n_estimators=200,min_child_weight=3) for d in (2,3,4) for lr in (0.05,0.1)]
    rows=[]
    for g in tqdm(GRID, desc='diag depth 2/3/4', unit='cfg'):
        ao,at,gp,af=diag_one(g,FEATS_DIAG); rows.append({**g,'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G=pd.DataFrame(rows); G.to_csv(GRIDC_CSV,index=False)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
print(f"At K={K_DIAG}:"); print(G[['max_depth','learning_rate','AP_oof','AP_train','gap','AP_f9']].to_string(index=False))
print(f"\nAP_train median = {G.AP_train.median():.3f} -> saturated. Judge on OOF + fold9 + inter-fold variance.")

### Block 7b.3d: 2D grid depth OPEN (2/3/4) + max-OOF selection + parsimony tiebreak

In [ ]:
# 2D K x config grid, depth OPEN (2/3/4) + selection = MAX OOF AP with a PARSIMONY TIEBREAK among true
# OOF-ties (smallest K within TIE_EPS, then best fold9). Same M3 rule as the combined detector (07a):
# train-gap dropped (saturated); guards = inter-fold OOF AP variance + fold9. NB: differs from M1/M2
# (1-SE / max-OOF-under-gap) on purpose - see header. Best STANDALONE model. fold10 + cross fold9 untouched.
G2_CSV = os.path.join(PROCESSED, f'm3_{TAG}_grid2d.csv')
kmax=len(dedup_list)
KS = sorted(set([k for k in [20,40,80,140,220,320] if k<kmax] + [K_auto, kmax]))
CFGS={}
for dep,lrs in [(2,(0.03,0.05)),(3,(0.03,0.05,0.1)),(4,(0.03,0.05,0.1))]:
    for lr in lrs:
        CFGS[f'd{dep}_lr{int(lr*100):02d}']=dict(max_depth=dep,learning_rate=lr,n_estimators=200,min_child_weight=3)
print(f"KS grid: {KS} | configs: {list(CFGS)}")
if os.path.exists(G2_CSV):
    G2=pd.read_csv(G2_CSV); print(f"[guard] {os.path.basename(G2_CSV)} reloaded.")
else:
    rows=[]
    for kk in tqdm(KS, desc='K x config (depth 2/3/4)', unit='K'):
        for cn,c in CFGS.items():
            ao,at,gp,af=diag_one(c,dedup_list[:kk]); rows.append({'K':kk,'cfg':cn,'depth':c['max_depth'],'AP_oof':ao,'AP_train':at,'gap':gp,'AP_f9':af})
    G2=pd.DataFrame(rows); G2.to_csv(G2_CSV,index=False)
pd.set_option('display.float_format',lambda x:f'{x:.4f}')
print(G2.sort_values('AP_oof',ascending=False).head(12).to_string(index=False))
def oof_perfold_ap(feats,**kw):
    oof=oof_same(feats,**kw); aps=[]
    for h in sorted(np.unique(folds)):
        msk=(folds==h)&(~np.isnan(oof))
        if y[msk].sum()>0: aps.append(average_precision_score(y[msk],oof[msk]))
    return np.array(aps)
maxoof=float(G2.AP_oof.max()); elig=G2[G2.AP_oof>=maxoof-TIE_EPS].copy()
print(f"\nMax OOF AP = {maxoof:.3f}; eligible (within {TIE_EPS}) = {len(elig)}:")
print(elig.sort_values(['K','AP_f9'],ascending=[True,False])[['K','cfg','depth','AP_oof','AP_f9']].to_string(index=False))
best=elig.sort_values(['K','AP_f9'],ascending=[True,False]).iloc[0]
K=int(best.K); CFGNAME=best['cfg']; FEATURES_K=dedup_list[:K]
CFG={**CFGS[CFGNAME],'subsample':0.8,'colsample_bytree':0.8,'reg_lambda':2.0}
pf=oof_perfold_ap(FEATURES_K,**CFGS[CFGNAME])
print(f"\n[selection] MAX OOF AP, depth open, parsimony tiebreak (TIE_EPS={TIE_EPS}).")
print(f">>> FINAL {TRAIN_SRC}: K={K}, cfg={CFGNAME} (depth {int(best.depth)}) | OOF AP={best.AP_oof:.3f} | fold9={best.AP_f9:.3f}")
print(f"  guards: per-fold OOF AP {pf.mean():.3f}+/-{pf.std():.3f} (range {pf.min():.3f}-{pf.max():.3f})")

### Block 7b.4: Same-dataset model + standardized evaluation (fold 9) + screening

In [ ]:
# Final same-dataset model + standardized evaluation (fold 9). Raw model (folds 1-8) for threshold/cross
# test; OOF native folds for F1-max threshold + Platt calibration (anti-shuffle); multi-seed. NB: fold 9
# has only 6-7 WPW -> high variance; the stable judge is the pooled OOF AP (folds 1-8).
from sklearn.linear_model import LogisticRegression as _LR
X8,X9=d8[FEATURES_K],f9[FEATURES_K]
def make_final(**kw):
    p=dict(**CFG,scale_pos_weight=spw8,eval_metric='aucpr',tree_method='hist',random_state=42,n_jobs=10); p.update(kw); return XGBClassifier(**p)
xgb_raw=make_final().fit(X8,y8); score9=xgb_raw.predict_proba(X9)[:,1]
ap_train=average_precision_score(y8,xgb_raw.predict_proba(X8)[:,1])
oof_raw=np.full(len(d8),np.nan)
for trm,vam in FOLD_MASKS:
    if y8[trm].sum()==0 or y8[vam].sum()==0: continue
    spw=(y8[trm]==0).sum()/max((y8[trm]==1).sum(),1)
    oof_raw[vam]=make_final(scale_pos_weight=spw).fit(X8[trm],y8[trm]).predict_proba(X8[vam])[:,1]
mask_oof=~np.isnan(oof_raw)
platt=_LR(max_iter=2000).fit(oof_raw[mask_oof].reshape(-1,1),y8[mask_oof])
score9_cal=platt.predict_proba(score9.reshape(-1,1))[:,1]
aps=[]; aucs=[]
for s in range(10):
    mm=make_final(random_state=s).fit(X8,y8); pp=mm.predict_proba(X9)[:,1]
    aps.append(average_precision_score(y9,pp)); aucs.append(roc_auc_score(y9,pp))
MULTISEED=dict(AP_mean=float(np.mean(aps)),AP_std=float(np.std(aps)),AUC_mean=float(np.mean(aucs)),AUC_std=float(np.std(aucs)))
ap_oof_final=average_precision_score(y8[mask_oof],oof_raw[mask_oof])
print(f"OOF AP folds 1-8 (stable judge) = {ap_oof_final:.3f}")
print(f"Multi-seed fold9 same: AP {np.mean(aps):.3f}+/-{np.std(aps):.3f} | AUC {np.mean(aucs):.3f}+/-{np.std(aucs):.3f}")
def prec_at_recall(yt,sc,target=0.8):
    order=np.argsort(-np.asarray(sc)); yy=np.asarray(yt)[order]
    tp=np.cumsum(yy); P=yy.sum(); rec=tp/P; prec=tp/np.arange(1,len(yy)+1)
    ok=np.where(rec>=target)[0]
    return (float(prec[ok[0]]),float(rec[ok[0]])) if len(ok) else (float('nan'),float('nan'))
p80,r80=prec_at_recall(y9,score9,0.8); print(f"Screening: precision@recall>=0.8 (fold9) = {p80:.3f} at recall {r80:.3f}")
SAME_METRICS=evaluate_standard(name=f'M3_{TAG}',y_oof=y8[mask_oof],score_oof=oof_raw[mask_oof],y_test=y9,score_test=score9,
    figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score9_cal,ap_train=ap_train,multiseed=MULTISEED,
    extra={'K':K,'POOL':POOL,'cfg':CFGNAME,'depth':int(best.depth),'OOF_AP_folds1_8':float(ap_oof_final),
           'precision_at_recall0.8_fold9':p80,'eval_type':'same_dataset_fold9','xgb_params':{**CFG}})

### Block 7b.4-cross: Cross-dataset transfer (batch effect)

In [ ]:
# Cross-dataset transfer: the TRAIN_SRC-trained model applied to CROSS_SRC folds 1-8 (cross 9-10 untouched).
# Expect AUC to hold (detection transfers) but AP to collapse (score scale does not transfer = batch effect).
score_cross = xgb_raw.predict_proba(dcx[FEATURES_K])[:,1]
score_cross_cal = platt.predict_proba(score_cross.reshape(-1,1))[:,1]
CROSS_METRICS=evaluate_standard(name=f'M3_{TAG}_CROSS_on_{CROSS_SRC}',
    y_oof=y8[mask_oof],score_oof=oof_raw[mask_oof],   # F1-max threshold on OOF same (consistent)
    y_test=ycx,score_test=score_cross,figures_dir=FIG,metrics_dir=METRICS,score_test_calibrated=score_cross_cal,
    extra={'eval_type':'cross_dataset','train_on':TRAIN_SRC,'test_on':f'{CROSS_SRC}_folds1_8'})
print("\n=== same vs cross (the batch-effect result) ===")
print(f"  SAME  ({TRAIN_SRC} fold9): AP {SAME_METRICS['AP']:.3f}  AUC {SAME_METRICS['AUC']:.3f}")
print(f"  CROSS ({CROSS_SRC} f1-8) : AP {CROSS_METRICS['AP']:.3f}  AUC {CROSS_METRICS['AUC']:.3f}")
print(f"  -> AP collapses ({SAME_METRICS['AP']:.3f} -> {CROSS_METRICS['AP']:.3f}) while AUC holds "
      f"({SAME_METRICS['AUC']:.3f} -> {CROSS_METRICS['AUC']:.3f}). Detection transfers, score scale does not.")
print("  -> justifies the COMBINED detector (07a) + percentile display.")

### Block 7b.5: Freeze (m3_ptbxl_* artifacts)

In [ ]:
# Freeze the single-dataset M3 detector (model + Platt + features + config with same & cross metrics).
# Fold 10 (both datasets) + fold 9 of the cross dataset NEVER touched.
import joblib
from datetime import datetime
MDIR=os.path.join(ROOT,'models',f'M3_{TAG}'); os.makedirs(MDIR,exist_ok=True)
joblib.dump(xgb_raw,os.path.join(MDIR,f'm3_{TAG}_xgb_raw.joblib'))
joblib.dump(platt,os.path.join(MDIR,f'm3_{TAG}_platt.joblib'))
pd.Series(FEATURES_K,name='feature').to_csv(os.path.join(MDIR,f'm3_{TAG}_features.csv'),index=False)
config={'model':f'M3_wavelet_{TAG}_pure',
 'train_on':f'{TRAIN_SRC} folds 1-8','val_same':f'{TRAIN_SRC} fold 9','cross_test':f'{CROSS_SRC} folds 1-8',
 'date_frozen':datetime.now().strftime('%Y-%m-%d %H:%M'),'version':'1.0','pool':POOL,
 'K':int(K),'cfg':CFGNAME,'depth':int(best.depth),'xgb_params':{**CFG,'scale_pos_weight':float(spw8)},'features':FEATURES_K,
 'representation':'wavelet LOCALIZATION (SWT db4+sym4+coif3 + WPT db4 L6 + DWT db4) + delineation-free QRS-localized families (QRS-mask, QRS-onset, polarity); energy dropped',
 'selection':'single-dataset gate |d|>0.3 & p_FDR<0.05 (BH) & IC95 excl 0 (no cross criterion); Spearman>0.95 dedup; localization+QRS pool; '
             'MAX OOF AP, depth open (2/3/4) + parsimony tiebreak among true OOF-ties (never selects on fold 9). Differs from M1/M2 (1-SE) on purpose - see header.',
 'metrics_same':SAME_METRICS,'metrics_cross':CROSS_METRICS,
 'batch_effect':f"AP same {SAME_METRICS['AP']:.3f} -> cross {CROSS_METRICS['AP']:.3f} (collapse); AUC same {SAME_METRICS['AUC']:.3f} -> cross {CROSS_METRICS['AUC']:.3f} (holds)",
 'notes':'Batch-effect diagnostic + standalone detector. F1-max threshold on OOF same. Fold 10 (both) + fold 9 cross NEVER touched.'}
with open(os.path.join(MDIR,f'm3_{TAG}_config.json'),'w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print(f"=== M3 {TAG} FROZEN -> {MDIR} ===")
print(f"  same OOF AP {ap_oof_final:.3f} | same fold9 AP {SAME_METRICS['AP']:.3f} AUC {SAME_METRICS['AUC']:.3f} | "
      f"cross AP {CROSS_METRICS['AP']:.3f} AUC {CROSS_METRICS['AUC']:.3f} | K={K} depth{int(best.depth)}")
print("  Fold 10 + fold 9 cross never touched.")

### Results & interpretation

**Standalone ptbxl:** the OOF AP (folds 1-8) is the stable figure; the fold-9 number rests on only 6-7 WPW and is high-variance. **Batch effect:** the model's cross-dataset AP on ningbo **collapses** while the AUC **holds** — detection (ranking) transfers, the score scale does not. This is the empirical justification for the **combined** detector (07a) and for a percentile-based score display. Fold 10 (both) + fold 9 of ningbo remain untouched.